# 🫁 Pneumonia Diagnostic Hub — AI Training Notebook
## Multi-Model CNN Ensemble + YOLOv8 Localization Engine
### Kaggle Training Pipeline — Production-Grade Export

---

> **Purpose**: Train 4 state-of-the-art CNN models (MobileNetV2, EfficientNetB0, DenseNet121, Xception) for pneumonia classification, plus YOLOv8 for radiological localization. All model outputs are calibrated to integrate directly with the deployed **Pneumonia Diagnostic Hub** Streamlit App.

---

## 📋 Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Dataset Exploration (EDA)](#2-eda)
3. [Preprocessing Pipeline](#3-preprocessing)
4. [CNN Model Training — 4 Architectures](#4-cnn-training)
5. [Model Evaluation & Comparison](#5-evaluation)
6. [Grad-CAM XAI Visualization](#6-gradcam)
7. [Ensemble Consensus Engine](#7-ensemble)
8. [YOLOv8 Pneumonia Localization](#8-yolo)
9. [Model Export & Streamlit Integration](#9-export)

---

### 🎯 Model → File Mapping (for Streamlit App)
| Model | Output File | Ensemble Weight |
|-------|-------------|-----------------|
| MobileNetV2 | `mobilenet_model.h5` | 25% |
| EfficientNetB0 | `efficientnet_model.h5` | 20% |
| DenseNet121 | `densenet121_model.h5` | 35% |
| Xception | `xception_model.h5` | 20% |
| YOLOv8n | `yolo_pneumonia/weights/best.pt` | — |

### 📦 Kaggle Datasets Required
Add these datasets to your notebook before running:
1. **`paultimothymooney/chest-xray-pneumonia`** — Classification training
2. **`rsna-pneumonia-detection-challenge`** — YOLO detection training


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  SECTION 1: ENVIRONMENT SETUP                               ║
# ╚══════════════════════════════════════════════════════════════╝

# Install additional packages (TF & CV2 are pre-installed on Kaggle)
!pip install ultralytics pydicom scikit-learn seaborn tabulate -q
!pip install tensorflow-addons -q 2>/dev/null || echo "TF-Addons not available, skipping"

print("✅ Package installation complete")


In [ ]:
import os
import gc
import sys
import time
import json
import math
import random
import shutil
import warnings
import itertools
from pathlib import Path
from glob import glob
from copy import deepcopy
from collections import Counter
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm

# TensorFlow
warnings.filterwarnings('ignore')
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, regularizers
from tensorflow.keras.applications import (
    MobileNetV2, EfficientNetB0, DenseNet121, Xception
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, CSVLogger, LambdaCallback
)
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc, f1_score,
    precision_score, recall_score, accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

# YOLO & DICOM
from ultralytics import YOLO
import pydicom

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Python:      {sys.version.split()[0]}")
print(f"TensorFlow:  {tf.__version__}")
print(f"Keras:       {keras.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"Pandas:      {pd.__version__}")
print(f"OpenCV:      {cv2.__version__}")


In [ ]:
# ─── GPU Detection & Memory Configuration ─────────────────────
print("=" * 55)
print("  GPU / HARDWARE ENVIRONMENT")
print("=" * 55)

gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')

if gpus:
    print(f"✅  {len(gpus)} GPU(s) detected:")
    for i, gpu in enumerate(gpus):
        print(f"    [{i}] {gpu.name}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅  Dynamic GPU memory growth enabled")
    except RuntimeError as e:
        print(f"⚠️  Memory growth config error: {e}")
else:
    print(f"⚠️  No GPU detected — using {len(cpus)} CPU(s)")
    print("   Training will be significantly slower")

# Mixed precision (speeds up training ~2x on T4)
try:
    from tensorflow.keras.mixed_precision import set_global_policy
    set_global_policy('mixed_float16')
    print("✅  Mixed precision (float16) enabled — ~2x speedup")
    USE_MIXED_PRECISION = True
except Exception as e:
    print(f"⚠️  Mixed precision not available: {e}")
    USE_MIXED_PRECISION = False

print("=" * 55)


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  MASTER CONFIGURATION                                       ║
# ║  ⚠️  These values MUST match your Streamlit app's config.py ║
# ╚══════════════════════════════════════════════════════════════╝

# ─── Image & Training Params ───────────────────────────────────
IMG_SIZE      = 128          # Matches SRS FR-021: 128x128 px
BATCH_SIZE    = 32
EPOCHS_PHASE1 = 10           # Frozen backbone — train head only
EPOCHS_PHASE2 = 25           # Fine-tuning — unfreeze last N layers

# ─── Class Labels (matches config.py CLASS_LABELS exactly) ─────
CLASS_NAMES   = ['NORMAL', 'PNEUMONIA']   # 0=NORMAL, 1=PNEUMONIA
CLASS_LABELS  = {0: 'NORMAL', 1: 'PNEUMONIA'}
NUM_CLASSES   = 2

# ─── Model Filenames (MUST match config.py AVAILABLE_MODELS) ───
MODEL_CONFIG = {
    'mobilenet': {
        'filename':          'mobilenet_model.h5',
        'display_name':      'MobileNetV2',
        'ensemble_weight':    0.25,
        'target_layer':      'out_relu',
        'unfreeze_last':      30,
        'parameters':        '3.5M',
        'lr_finetune':        1e-5,
    },
    'efficientnet': {
        'filename':          'efficientnet_model.h5',
        'display_name':      'EfficientNetB0',
        'ensemble_weight':    0.20,
        'target_layer':      'top_activation',
        'unfreeze_last':      20,
        'parameters':        '5.3M',
        'lr_finetune':        5e-6,
    },
    'densenet121': {
        'filename':          'densenet121_model.h5',
        'display_name':      'DenseNet121',
        'ensemble_weight':    0.35,
        'target_layer':      'relu',        # Last BN activation before GAP
        'unfreeze_last':      40,            # Unfreeze last dense block
        'parameters':        '8.1M',
        'lr_finetune':        1e-5,
    },
    'xception': {
        'filename':          'xception_model.h5',
        'display_name':      'Xception',
        'ensemble_weight':    0.20,
        'target_layer':      'block14_sepconv2_act',  # Last sep-conv activation
        'unfreeze_last':      26,            # Unfreeze exit flow
        'parameters':        '22.9M',
        'lr_finetune':        5e-6,
    },
}

# ─── Paths ─────────────────────────────────────────────────────
# ─── Auto-detect chest-xray-pneumonia dataset root ─────────────────
# Handles different Kaggle dataset slug mount points automatically
_POSSIBLE_ROOTS = [
    Path('/kaggle/input/chest-xray-pneumonia/chest_xray'),
    Path('/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray'),
    Path('/kaggle/input/chest-xray-pneumonia'),
    Path('/kaggle/input/chestxray2013/chest_xray'),
    Path('/kaggle/input/chest-x-ray-pneumonia/chest_xray'),
    Path('/kaggle/input/paul-mooney-chest-xray-pneumonia/chest_xray'),
]

DATASET_BASE = None
for _p in _POSSIBLE_ROOTS:
    if (_p / 'train' / 'NORMAL').exists() or (_p / 'train').exists():
        DATASET_BASE = _p
        break

if DATASET_BASE is None:
    # Last resort: walk /kaggle/input/ looking for a 'NORMAL' folder
    for _candidate in Path('/kaggle/input').rglob('NORMAL'):
        if _candidate.parent.name == 'train':
            DATASET_BASE = _candidate.parent.parent
            break

if DATASET_BASE is None:
    print("ERROR: chest-xray-pneumonia dataset not found!")
    print("Available /kaggle/input/ contents:")
    for _d in sorted(Path('/kaggle/input').iterdir()):
        print(f"  {_d}")
    raise FileNotFoundError(
        "Dataset not mounted. Go to Kaggle notebook Settings -> "
        "Add Data -> search 'chest-xray-pneumonia' by paultimothymooney"
    )
else:
    print(f"Dataset root auto-detected: {DATASET_BASE}")
RSNA_DIR      = Path('/kaggle/input/rsna-pneumonia-detection-challenge')
OUTPUT_DIR    = Path('/kaggle/working')
YOLO_DATA_DIR = OUTPUT_DIR / 'yolo_dataset'
LOG_DIR       = OUTPUT_DIR / 'logs'

# Create output directories
for d in [OUTPUT_DIR, LOG_DIR, YOLO_DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Dataset split paths
SPLIT_DIRS = {
    'train': DATASET_BASE / 'train',
    'val':   DATASET_BASE / 'val',
    'test':  DATASET_BASE / 'test',
}

print("✅  Configuration loaded")
print(f"   Image size  : {IMG_SIZE}×{IMG_SIZE}")
print(f"   Batch size  : {BATCH_SIZE}")
print(f"   Phase 1     : {EPOCHS_PHASE1} epochs (frozen)")
print(f"   Phase 2     : {EPOCHS_PHASE2} epochs (fine-tune)")
print(f"   Output dir  : {OUTPUT_DIR}")


---
## 2. 📊 Dataset Exploration (EDA)
### Dataset: Chest X-Ray Images (Pneumonia) — Paul Mooney, Kaggle
**Reference**: SRS §9.5 — Base Dataset: `paultimothymooney/chest-xray-pneumonia`


In [ ]:
# ─── Dataset Verification & Statistics ────────────────────────

# Supported image extensions (covers all common Kaggle naming)
IMG_EXTS = ('*.jpeg', '*.jpg', '*.png', '*.JPEG', '*.JPG', '*.PNG')

def count_images(directory: Path, exts=IMG_EXTS) -> int:
    """Count images in a directory, returns 0 if directory doesn't exist."""
    if not directory.exists():
        return 0
    total = 0
    for ext in exts:
        total += len(list(directory.glob(ext)))
    return total

# ─── Debug: show what's actually in the dataset root ─────────────
print(f"Dataset root : {DATASET_BASE}")
print(f"Root exists  : {DATASET_BASE.exists()}")
if DATASET_BASE.exists():
    top_level = sorted(DATASET_BASE.iterdir())
    print(f"Contents     : {[d.name for d in top_level]}")

print()
print("=" * 65)
print("  DATASET INVENTORY")
print("=" * 65)
print(f"{'Split':<10} {'NORMAL':>10} {'PNEUMONIA':>12} {'Total':>10} {'Ratio':>10}")
print("-" * 65)

stats = {}
for split_name, split_dir in SPLIT_DIRS.items():
    norm_dir = split_dir / 'NORMAL'
    pneu_dir = split_dir / 'PNEUMONIA'

    n = count_images(norm_dir)
    p = count_images(pneu_dir)
    t = n + p
    r = p / n if n > 0 else 0.0

    stats[split_name] = {'NORMAL': n, 'PNEUMONIA': p, 'total': t, 'ratio': r}

    # Show path status if empty
    norm_ok = "[OK]" if norm_dir.exists() else "[MISSING]"
    pneu_ok = "[OK]" if pneu_dir.exists() else "[MISSING]"
    print(f"{split_name.capitalize():<10} {n:>10,} {p:>12,} {t:>10,} {r:>9.2f}x"
          + (f"  NORMAL{norm_ok} PNEUMONIA{pneu_ok}" if t == 0 else ""))

print("=" * 65)
print()

train_n = stats['train']['NORMAL']
train_p = stats['train']['PNEUMONIA']

# Guard against empty dataset
if train_n == 0 or train_p == 0:
    print("ERROR: Dataset appears empty or not mounted correctly!")
    print()
    print("TROUBLESHOOTING STEPS:")
    print("  1. Click 'Add Data' in the right sidebar of your Kaggle notebook")
    print("  2. Search for: chest-xray-pneumonia  (by paultimothymooney)")
    print("  3. Add it and re-run all cells")
    print()
    print("Available datasets currently mounted:")
    for d in sorted(Path('/kaggle/input').iterdir()):
        print(f"  /kaggle/input/{d.name}/")
        for sub in sorted(d.iterdir())[:5]:
            print(f"    {sub.name}")
    raise RuntimeError(
        "Dataset not found. Add 'paultimothymooney/chest-xray-pneumonia' "
        "via Kaggle notebook Settings -> Add Data."
    )

print(f"Class imbalance: PNEUMONIA is {train_p/train_n:.1f}x more frequent than NORMAL")
print("  -> Will apply class_weight='balanced' during training")
print(f"Dataset root: {DATASET_BASE}")

In [ ]:
# ─── EDA — Class Distribution Visualization ───────────────────

fig = plt.figure(figsize=(20, 6))
fig.suptitle('Chest X-Ray Pneumonia Dataset — Exploratory Analysis',
             fontsize=17, fontweight='bold', y=1.01)

# Plot 1: Grouped bar chart
ax1 = fig.add_subplot(1, 3, 1)
splits = ['train', 'val', 'test']
x = np.arange(len(splits))
w = 0.35
n_counts = [stats[s]['NORMAL']    for s in splits]
p_counts = [stats[s]['PNEUMONIA'] for s in splits]

b1 = ax1.bar(x - w/2, n_counts, w, label='NORMAL',    color='#27AE60', alpha=0.85, edgecolor='white', linewidth=1.2)
b2 = ax1.bar(x + w/2, p_counts, w, label='PNEUMONIA', color='#E74C3C', alpha=0.85, edgecolor='white', linewidth=1.2)
ax1.set_xlabel('Dataset Split', fontsize=11)
ax1.set_ylabel('Image Count',   fontsize=11)
ax1.set_title('Class Distribution', fontsize=13, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels([s.capitalize() for s in splits])
ax1.legend(fontsize=10); ax1.grid(axis='y', alpha=0.3)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., h + 30, f'{h:,}',
             ha='center', va='bottom', fontsize=8, fontweight='bold')

# Plot 2: Pie chart (training split)
ax2 = fig.add_subplot(1, 3, 2)
colors   = ['#27AE60', '#E74C3C']
explode  = (0.05, 0.05)
wedges, texts, auto = ax2.pie(
    [train_n, train_p], labels=CLASS_NAMES, colors=colors,
    autopct='%1.1f%%', startangle=90, explode=explode,
    shadow=True, textprops={'fontsize': 11}
)
auto[0].set_fontsize(12); auto[1].set_fontsize(12)
ax2.set_title('Training Set Balance', fontsize=13, fontweight='bold')

# Plot 3: Total images per split
ax3 = fig.add_subplot(1, 3, 3)
totals = [stats[s]['total'] for s in splits]
bars   = ax3.bar([s.capitalize() for s in splits], totals,
                  color=['#3498DB', '#9B59B6', '#F39C12'],
                  edgecolor='white', linewidth=1.2, alpha=0.85)
ax3.set_ylabel('Total Images', fontsize=11)
ax3.set_title('Total Images per Split', fontsize=13, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, totals):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
             f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 EDA plot saved to /kaggle/working/eda_class_distribution.png")


In [ ]:
# ─── EDA — Sample Images Grid ─────────────────────────────────

def load_sample_images(class_dir: Path, n: int = 4) -> List:
    files = list(class_dir.glob('*.jpeg'))[:n]
    if not files:
        files = list(class_dir.glob('*.jpg'))[:n]
    images = []
    for f in files:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            images.append((img, f.name, img.shape))
    return images

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
fig.suptitle('Sample Chest X-Ray Images — Kaggle Dataset',
             fontsize=16, fontweight='bold', y=1.01)

row_labels = ['NORMAL', 'PNEUMONIA']
row_colors = ['#27AE60', '#E74C3C']

for row_idx, cls in enumerate(CLASS_NAMES):
    cls_dir  = SPLIT_DIRS['train'] / cls
    samples  = load_sample_images(cls_dir, n=5)
    axes[row_idx][0].set_ylabel(cls, fontsize=13, fontweight='bold',
                                 color=row_colors[row_idx], rotation=90)
    for col_idx in range(5):
        ax = axes[row_idx][col_idx]
        if col_idx < len(samples):
            img, fname, shape = samples[col_idx]
            ax.imshow(img, cmap='gray', aspect='auto')
            ax.set_title(f'{shape[1]}×{shape[0]}px', fontsize=8, color='#666')
        else:
            ax.set_visible(False)
            continue
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor(row_colors[row_idx])
            spine.set_linewidth(2)
            spine.set_visible(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── EDA — Image Dimension Analysis ─────────────────────────-

print("Analyzing image dimensions (sample of 200 per class)...")

dim_data = []
for cls in CLASS_NAMES:
    cls_dir = SPLIT_DIRS['train'] / cls
    files = list(cls_dir.glob('*.jpeg'))[:200] + list(cls_dir.glob('*.jpg'))[:200]
    for f in files[:200]:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            h, w = img.shape
            dim_data.append({'Class': cls, 'Height': h, 'Width': w,
                             'AspectRatio': w/h, 'Pixels': h*w})

df_dims = pd.DataFrame(dim_data)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Image Dimension Statistics', fontsize=15, fontweight='bold')

# Height distribution
axes[0].hist(df_dims[df_dims['Class']=='NORMAL']['Height'],
             bins=30, alpha=0.7, label='NORMAL',    color='#27AE60')
axes[0].hist(df_dims[df_dims['Class']=='PNEUMONIA']['Height'],
             bins=30, alpha=0.7, label='PNEUMONIA', color='#E74C3C')
axes[0].set_xlabel('Height (px)'); axes[0].set_ylabel('Count')
axes[0].set_title('Image Height Distribution'); axes[0].legend()
axes[0].axvline(IMG_SIZE, color='blue', linestyle='--', lw=2, label=f'Target: {IMG_SIZE}px')
axes[0].legend()

# Width distribution
axes[1].hist(df_dims[df_dims['Class']=='NORMAL']['Width'],
             bins=30, alpha=0.7, label='NORMAL',    color='#27AE60')
axes[1].hist(df_dims[df_dims['Class']=='PNEUMONIA']['Width'],
             bins=30, alpha=0.7, label='PNEUMONIA', color='#E74C3C')
axes[1].set_xlabel('Width (px)'); axes[1].set_ylabel('Count')
axes[1].set_title('Image Width Distribution'); axes[1].legend()

# Aspect ratio
axes[2].hist(df_dims[df_dims['Class']=='NORMAL']['AspectRatio'],
             bins=20, alpha=0.7, label='NORMAL',    color='#27AE60')
axes[2].hist(df_dims[df_dims['Class']=='PNEUMONIA']['AspectRatio'],
             bins=20, alpha=0.7, label='PNEUMONIA', color='#E74C3C')
axes[2].set_xlabel('Aspect Ratio (W/H)'); axes[2].set_ylabel('Count')
axes[2].set_title('Aspect Ratio Distribution'); axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📐 Dimension Summary:")
print(df_dims.groupby('Class')[['Height','Width','AspectRatio']].describe().round(1).to_string())
print(f"\n→ All images will be resized to {IMG_SIZE}×{IMG_SIZE} px (SRS FR-021)")


---
## 3. 🔄 Preprocessing Pipeline

**Exact pipeline matching SRS §4.3 (FR-020 to FR-025) and `core/preprocessor.py`:**
```
Raw Image → Grayscale → Resize 128×128 (INTER_AREA) → 3-ch Stack → /255.0 → (1,128,128,3)
```

> ℹ️ **ImageDataGenerator uses `color_mode='rgb'`** — chest X-ray JPEGs load as pseudo-RGB (R=G=B=gray), perfectly replicating the inference preprocessing pipeline.


In [ ]:
# ─── Class Weights (handle imbalance from SRS §9.5) ──────────

y_all = ([0] * stats['train']['NORMAL'] +
         [1] * stats['train']['PNEUMONIA'])
cw    = compute_class_weight('balanced',
                              classes=np.array([0, 1]),
                              y=np.array(y_all))
CLASS_WEIGHT = {0: float(cw[0]), 1: float(cw[1])}

print("Class Weights (to handle class imbalance):")
print(f"  NORMAL    (0): {CLASS_WEIGHT[0]:.4f}")
print(f"  PNEUMONIA (1): {CLASS_WEIGHT[1]:.4f}")

# ─── Augmentation Strategy ────────────────────────────────────
# Training: with augmentation
TRAIN_DATAGEN = ImageDataGenerator(
    rescale            = 1./255,           # Normalize [0,1] — SRS FR-023
    rotation_range     = 10,               # ±10° rotation
    width_shift_range  = 0.10,
    height_shift_range = 0.10,
    shear_range        = 0.10,
    zoom_range         = 0.10,
    horizontal_flip    = True,             # Valid for CXR (symmetric)
    brightness_range   = [0.80, 1.20],    # Simulate exposure variation
    fill_mode          = 'nearest'
)

# Validation/Test: rescale only, NO augmentation
EVAL_DATAGEN = ImageDataGenerator(rescale=1./255)

print("\n✅ Data augmentation configured:")
print("   rotation ±10°, shift 10%, zoom 10%, h-flip, brightness ±20%")


In [ ]:
# ─── Build Data Generators ────────────────────────────────────

TARGET_SIZE = (IMG_SIZE, IMG_SIZE)    # (128, 128)

train_gen = TRAIN_DATAGEN.flow_from_directory(
    SPLIT_DIRS['train'],
    target_size  = TARGET_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    color_mode   = 'rgb',              # Grayscale JPEG → pseudo-RGB (R=G=B)
    shuffle      = True,
    seed         = SEED
)

val_gen = EVAL_DATAGEN.flow_from_directory(
    SPLIT_DIRS['val'],
    target_size  = TARGET_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    color_mode   = 'rgb',
    shuffle      = False
)

test_gen = EVAL_DATAGEN.flow_from_directory(
    SPLIT_DIRS['test'],
    target_size  = TARGET_SIZE,
    batch_size   = BATCH_SIZE,
    class_mode   = 'categorical',
    color_mode   = 'rgb',
    shuffle      = False
)

print(f"Class indices : {train_gen.class_indices}")
print(f"  → 0=NORMAL (index matches config.py CLASS_LABELS[0]='NORMAL')")
print(f"  → 1=PNEUMONIA (index matches config.py CLASS_LABELS[1]='PNEUMONIA')")

print(f"\nGenerator Statistics:")
print(f"  Train  : {train_gen.samples:,} images, {len(train_gen)} steps/epoch")
print(f"  Val    : {val_gen.samples:,} images, {len(val_gen)} steps/epoch")
print(f"  Test   : {test_gen.samples:,} images, {len(test_gen)} steps/epoch")

# Verify batch shape
x_batch, y_batch = next(iter(train_gen))
print(f"\nBatch tensor shape : {x_batch.shape}")   # Should be (32, 128, 128, 3)
print(f"Label tensor shape  : {y_batch.shape}")   # Should be (32, 2)
print(f"Pixel value range   : [{x_batch.min():.3f}, {x_batch.max():.3f}]")


In [ ]:
# ─── Augmentation Visualization ───────────────────────────────

sample_img_path = list((SPLIT_DIRS['train'] / 'PNEUMONIA').glob('*.jpeg'))[0]
orig_img = cv2.imread(str(sample_img_path))
orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
orig_resized = cv2.resize(orig_img, (IMG_SIZE, IMG_SIZE))
x_arr = np.expand_dims(orig_resized, axis=0).astype(np.float32) / 255.

aug_gen = TRAIN_DATAGEN.flow(x_arr, batch_size=1, seed=SEED)

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle('Data Augmentation Samples (PNEUMONIA X-Ray)', fontsize=14, fontweight='bold')
axes[0][0].imshow(orig_resized)
axes[0][0].set_title('Original', fontsize=10, fontweight='bold')
axes[0][0].axis('off')

for i in range(1, 10):
    aug = next(aug_gen)[0]
    row, col = divmod(i, 5)
    axes[row][col].imshow(np.clip(aug, 0, 1))
    axes[row][col].set_title(f'Augmented #{i}', fontsize=9)
    axes[row][col].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'augmentation_samples.png', dpi=150, bbox_inches='tight')
plt.show()


---
## 4. 🏗️ CNN Model Training — 4 Architectures

### Training Strategy
- **Phase 1** (frozen backbone): Train only the custom classification head for `EPOCHS_PHASE1` epochs with `lr=1e-3`
- **Phase 2** (fine-tuning): Unfreeze the last N layers of the backbone, train with `lr=1e-5` to `5e-6`
- **Callbacks**: EarlyStopping, ModelCheckpoint (best val_accuracy), ReduceLROnPlateau
- **Class weights**: Applied to handle PNEUMONIA/NORMAL imbalance


In [ ]:
# ─── Model Architecture Builder ──────────────────────────────

def build_transfer_model(model_id: str,
                          input_shape: Tuple = (IMG_SIZE, IMG_SIZE, 3)
                         ) -> Tuple[tf.keras.Model, tf.keras.Model]:
    """
    Build a transfer learning model with custom pneumonia classification head.

    Architecture:
      ImageNet Backbone (frozen) → GlobalAveragePooling2D
      → BatchNorm → Dense(512, relu) → Dropout(0.5)
      → Dense(256, relu) → Dropout(0.3)
      → Dense(2, softmax, float32)   ← float32 output required for mixed precision

    Returns: (full_model, backbone_base)
    """
    backbone_constructors = {
        'mobilenet':    MobileNetV2,
        'efficientnet': EfficientNetB0,
        'densenet121':  DenseNet121,
        'xception':     Xception,
    }

    assert model_id in backbone_constructors, f"Unknown model_id: {model_id}"
    BackboneCls = backbone_constructors[model_id]

    # Load ImageNet pre-trained backbone (include_top=False)
    base = BackboneCls(
        weights       = 'imagenet',
        include_top   = False,
        input_shape   = input_shape
    )
    base.trainable = False   # Freeze for Phase 1

    # Custom classification head
    inputs = keras.Input(shape=input_shape, name='input_image')
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     name='dense_512')(x)
    x = layers.Dropout(0.5, name='drop_1')(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     name='dense_256')(x)
    x = layers.Dropout(0.3, name='drop_2')(x)
    # dtype=float32 ensures output is float32 even with mixed_float16 policy
    outputs = layers.Dense(NUM_CLASSES, activation='softmax',
                           dtype='float32', name='predictions')(x)

    model = keras.Model(inputs=inputs, outputs=outputs,
                        name=f'{model_id}_pneumonia')

    print(f"\n{'─'*55}")
    print(f"  Model : {MODEL_CONFIG[model_id]['display_name']}")
    print(f"  Params: {model.count_params():,} total")
    trainable = sum(p.numpy() for p in model.trainable_variables)
    print(f"  Train : {model.count_params() - int(sum(l.count_params() for l in base.layers)):,} (head only, Phase 1)")
    print(f"{'─'*55}")

    return model, base


def get_callbacks(model_id: str, phase: int) -> list:
    """Create standard callbacks for model training."""
    return [
        EarlyStopping(
            monitor             = 'val_loss',
            patience            = 5,
            restore_best_weights= True,
            verbose             = 1
        ),
        ModelCheckpoint(
            filepath            = str(OUTPUT_DIR / f'{model_id}_phase{phase}_best.h5'),
            monitor             = 'val_accuracy',
            save_best_only      = True,
            verbose             = 0
        ),
        ReduceLROnPlateau(
            monitor             = 'val_loss',
            factor              = 0.5,
            patience            = 3,
            min_lr              = 1e-8,
            verbose             = 1
        ),
        CSVLogger(
            filename            = str(LOG_DIR / f'{model_id}_phase{phase}.csv'),
            append              = False
        ),
    ]


def compile_model(model, lr: float):
    """Compile model with Adam optimizer and standard metrics."""
    model.compile(
        optimizer = optimizers.Adam(learning_rate=lr),
        loss      = 'categorical_crossentropy',
        metrics   = [
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ]
    )


print("✅ Model builder utilities defined")


In [ ]:
# ─── Two-Phase Training Engine ────────────────────────────────

def train_two_phase(model_id: str,
                    train_gen, val_gen,
                    class_weight: Dict,
                    epochs_p1: int = EPOCHS_PHASE1,
                    epochs_p2: int = EPOCHS_PHASE2
                   ) -> Dict[str, Any]:
    """
    Two-phase training for a single model backbone:
      Phase 1 — Frozen backbone, train head only (high LR)
      Phase 2 — Unfreeze last N layers, fine-tune (low LR)

    Returns: dict with model, histories, best metrics
    """
    cfg = MODEL_CONFIG[model_id]
    display = cfg['display_name']
    print(f"\n{'═'*60}")
    print(f"  🏋️  TRAINING: {display} ({model_id})")
    print(f"  Ensemble weight: {cfg['ensemble_weight']*100:.0f}%  |  "
          f"Target layer: {cfg['target_layer']}")
    print(f"{'═'*60}")

    model, base = build_transfer_model(model_id)

    # ── PHASE 1: Train classification head ──────────────────
    print(f"\n  ► Phase 1: Frozen backbone — {epochs_p1} epochs, lr=1e-3")
    compile_model(model, lr=1e-3)

    h1 = model.fit(
        train_gen,
        steps_per_epoch  = len(train_gen),
        epochs           = epochs_p1,
        validation_data  = val_gen,
        validation_steps = len(val_gen),
        class_weight     = class_weight,
        callbacks        = get_callbacks(model_id, phase=1),
        verbose          = 1
    )

    p1_best_val_acc = max(h1.history['val_accuracy'])
    print(f"  ✔ Phase 1 best val_accuracy: {p1_best_val_acc:.4f}")

    # Reset generators for Phase 2
    train_gen.reset()
    val_gen.reset()

    # ── PHASE 2: Fine-tune unfrozen layers ───────────────────
    n_unfreeze = cfg['unfreeze_last']
    lr_ft      = cfg['lr_finetune']

    print(f"\n  ► Phase 2: Unfreeze last {n_unfreeze} backbone layers — "
          f"{epochs_p2} epochs, lr={lr_ft:.0e}")

    # Selectively unfreeze
    base.trainable = True
    total_layers   = len(base.layers)
    freeze_until   = total_layers - n_unfreeze
    for i, layer in enumerate(base.layers):
        layer.trainable = (i >= freeze_until)

    # Re-compile with lower learning rate
    compile_model(model, lr=lr_ft)

    h2 = model.fit(
        train_gen,
        steps_per_epoch  = len(train_gen),
        epochs           = epochs_p2,
        validation_data  = val_gen,
        validation_steps = len(val_gen),
        class_weight     = class_weight,
        callbacks        = get_callbacks(model_id, phase=2),
        verbose          = 1
    )

    p2_best_val_acc = max(h2.history['val_accuracy'])
    print(f"  ✔ Phase 2 best val_accuracy: {p2_best_val_acc:.4f}")

    # ── Save final model (exact filename for Streamlit app) ──
    save_path = OUTPUT_DIR / cfg['filename']
    model.save(str(save_path))
    size_mb   = save_path.stat().st_size / 1e6
    print(f"\n  💾 Saved: {save_path.name}  ({size_mb:.1f} MB)")

    # Garbage collect
    del base
    gc.collect()
    tf.keras.backend.clear_session()

    return {
        'model_id':    model_id,
        'model':       model,
        'history_p1':  h1.history,
        'history_p2':  h2.history,
        'save_path':   save_path,
        'best_val_acc': max(p1_best_val_acc, p2_best_val_acc),
    }


print("✅ Two-phase training engine ready")


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  TRAIN ALL 4 MODELS                                         ║
# ║  ⏱️  Estimated time: ~30-60 min on Kaggle T4 GPU            ║
# ╚══════════════════════════════════════════════════════════════╝

TRAINING_RESULTS = {}
MODEL_ORDER = ['mobilenet', 'efficientnet', 'densenet121', 'xception']

for model_id in MODEL_ORDER:
    # Reset generators before each model
    train_gen.reset()
    val_gen.reset()

    result = train_two_phase(
        model_id     = model_id,
        train_gen    = train_gen,
        val_gen      = val_gen,
        class_weight = CLASS_WEIGHT,
        epochs_p1    = EPOCHS_PHASE1,
        epochs_p2    = EPOCHS_PHASE2,
    )
    TRAINING_RESULTS[model_id] = result

    # Force clear GPU memory between models
    tf.keras.backend.clear_session()
    gc.collect()

print("\n" + "═"*60)
print("  ✅  ALL 4 MODELS TRAINED SUCCESSFULLY")
print("═"*60)
print(f"{'Model':<20} {'Best Val Acc':>14} {'File':>25}")
print("─"*60)
for mid, res in TRAINING_RESULTS.items():
    print(f"{MODEL_CONFIG[mid]['display_name']:<20} "
          f"{res['best_val_acc']:>13.4f}  "
          f"{res['save_path'].name:>25}")


---
## 5. 📈 Model Evaluation & Comparison


In [ ]:
# ─── Training History Plots ───────────────────────────────────

def merge_histories(h1: dict, h2: dict) -> dict:
    """Merge Phase 1 and Phase 2 histories for clean plotting."""
    merged = {}
    for key in h1:
        merged[key] = h1[key] + h2.get(key, [])
    return merged


fig, axes = plt.subplots(4, 2, figsize=(18, 28))
fig.suptitle('Training Histories — All 4 Models', fontsize=16, fontweight='bold', y=1.01)

colors = {'train': '#3498DB', 'val': '#E74C3C'}

for row_idx, model_id in enumerate(MODEL_ORDER):
    res     = TRAINING_RESULTS[model_id]
    history = merge_histories(res['history_p1'], res['history_p2'])
    n_p1    = len(res['history_p1']['accuracy'])
    display = MODEL_CONFIG[model_id]['display_name']

    # Accuracy
    ax_acc = axes[row_idx][0]
    ax_acc.plot(history['accuracy'],     color=colors['train'], lw=2, label='Train Accuracy')
    ax_acc.plot(history['val_accuracy'], color=colors['val'],   lw=2, label='Val Accuracy')
    ax_acc.axvline(n_p1-1, color='gray', lw=1.5, linestyle='--', alpha=0.7, label=f'P1→P2 ({n_p1})')
    ax_acc.set_title(f'{display} — Accuracy', fontsize=12, fontweight='bold')
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy')
    ax_acc.legend(fontsize=9); ax_acc.grid(alpha=0.3)
    ax_acc.set_ylim([0.5, 1.01])

    # Loss
    ax_loss = axes[row_idx][1]
    ax_loss.plot(history['loss'],     color=colors['train'], lw=2, label='Train Loss')
    ax_loss.plot(history['val_loss'], color=colors['val'],   lw=2, label='Val Loss')
    ax_loss.axvline(n_p1-1, color='gray', lw=1.5, linestyle='--', alpha=0.7, label=f'P1→P2 ({n_p1})')
    ax_loss.set_title(f'{display} — Loss', fontsize=12, fontweight='bold')
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss')
    ax_loss.legend(fontsize=9); ax_loss.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_histories.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Training histories saved")


In [ ]:
# ─── Reload Best Models & Evaluate on Test Set ───────────────

# Custom layer handler for quantization_config compatibility
def safe_load_h5(filepath: str) -> tf.keras.Model:
    """Load .h5 model with Keras 2/3 compatibility (matches app's _safe_load_model)."""
    try:
        return tf.keras.models.load_model(
            filepath, compile=False,
            custom_objects={'DepthwiseConv2D': tf.keras.layers.DepthwiseConv2D}
        )
    except Exception:
        return tf.keras.models.load_model(filepath, compile=False)


LOADED_MODELS = {}
for model_id in MODEL_ORDER:
    path = OUTPUT_DIR / MODEL_CONFIG[model_id]['filename']
    if path.exists():
        print(f"Loading {MODEL_CONFIG[model_id]['display_name']} from {path.name}...")
        LOADED_MODELS[model_id] = safe_load_h5(str(path))
    else:
        # Fallback: use the in-memory model
        LOADED_MODELS[model_id] = TRAINING_RESULTS[model_id]['model']
        print(f"Using in-memory model for {model_id}")

print(f"\n✅ {len(LOADED_MODELS)} models loaded for evaluation")


In [ ]:
# ─── Comprehensive Test Set Evaluation ────────────────────────

def evaluate_model_on_test(model, model_id: str, test_gen) -> Dict[str, Any]:
    """Run full evaluation: accuracy, AUC, F1, precision, recall, confusion matrix."""
    test_gen.reset()

    # Predict all test images
    y_pred_proba = model.predict(test_gen, steps=len(test_gen), verbose=0)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    y_true       = test_gen.classes

    # Metrics
    acc       = accuracy_score(y_true, y_pred)
    f1_macro  = f1_score(y_true, y_pred, average='macro')
    f1_binary = f1_score(y_true, y_pred, pos_label=1)   # PNEUMONIA F1
    prec      = precision_score(y_true, y_pred, pos_label=1)
    rec       = recall_score(y_true, y_pred, pos_label=1)
    try:
        auc_score = roc_auc_score(y_true, y_pred_proba[:, 1])
    except Exception:
        auc_score = 0.0

    cm = confusion_matrix(y_true, y_pred)

    return {
        'model_id':        model_id,
        'display_name':    MODEL_CONFIG[model_id]['display_name'],
        'accuracy':        acc,
        'f1_macro':        f1_macro,
        'f1_pneumonia':    f1_binary,
        'precision':       prec,
        'recall':          rec,
        'auc':             auc_score,
        'confusion_matrix': cm,
        'y_true':          y_true,
        'y_pred':          y_pred,
        'y_pred_proba':    y_pred_proba,
    }


EVAL_RESULTS = {}
for model_id in MODEL_ORDER:
    print(f"Evaluating {MODEL_CONFIG[model_id]['display_name']}...", end=' ')
    res = evaluate_model_on_test(LOADED_MODELS[model_id], model_id, test_gen)
    EVAL_RESULTS[model_id] = res
    print(f"Acc={res['accuracy']:.4f}  AUC={res['auc']:.4f}  F1={res['f1_pneumonia']:.4f}")

print("\n✅ Evaluation complete")


In [ ]:
# ─── Model Comparison Table ───────────────────────────────────

rows = []
for model_id in MODEL_ORDER:
    res = EVAL_RESULTS[model_id]
    rows.append({
        'Model':              MODEL_CONFIG[model_id]['display_name'],
        'Accuracy (%)':       f"{res['accuracy']*100:.2f}",
        'AUC':                f"{res['auc']:.4f}",
        'F1 (Pneumonia)':     f"{res['f1_pneumonia']:.4f}",
        'Precision':          f"{res['precision']:.4f}",
        'Recall':             f"{res['recall']:.4f}",
        'F1 (Macro)':         f"{res['f1_macro']:.4f}",
        'Ensemble Weight':    f"{MODEL_CONFIG[model_id]['ensemble_weight']*100:.0f}%",
    })

df_compare = pd.DataFrame(rows)
print("\n" + "═"*85)
print("  MODEL COMPARISON — TEST SET")
print("═"*85)
print(df_compare.to_string(index=False))
print("═"*85)

# Save as CSV
df_compare.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
print("\n📊 model_comparison.csv saved")


In [ ]:
# ─── Confusion Matrices ───────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Confusion Matrices — Test Set', fontsize=16, fontweight='bold')

for idx, model_id in enumerate(MODEL_ORDER):
    ax  = axes[idx // 2][idx % 2]
    res = EVAL_RESULTS[model_id]
    cm  = res['confusion_matrix']
    acc = res['accuracy']

    # Normalize to percentages
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    im = ax.imshow(cm_pct, interpolation='nearest', cmap='Blues', vmin=0, vmax=100)
    ax.set_title(f"{MODEL_CONFIG[model_id]['display_name']}
"
                 f"Accuracy: {acc*100:.2f}%",
                 fontsize=12, fontweight='bold')

    tick_marks = np.arange(len(CLASS_NAMES))
    ax.set_xticks(tick_marks); ax.set_xticklabels(CLASS_NAMES, fontsize=11)
    ax.set_yticks(tick_marks); ax.set_yticklabels(CLASS_NAMES, fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)

    # Annotate cells
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            color = 'white' if cm_pct[i, j] > 50 else 'black'
            ax.text(j, i,
                    f"{cm[i,j]}\n({cm_pct[i,j]:.1f}%)",
                    ha='center', va='center',
                    color=color, fontsize=12, fontweight='bold')

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── ROC Curves ───────────────────────────────────────────────

MODEL_COLORS = {
    'mobilenet':    '#3498DB',
    'efficientnet': '#27AE60',
    'densenet121':  '#E74C3C',
    'xception':     '#F39C12',
}

fig, ax = plt.subplots(1, 1, figsize=(9, 8))

for model_id in MODEL_ORDER:
    res  = EVAL_RESULTS[model_id]
    fpr, tpr, _ = roc_curve(res['y_true'], res['y_pred_proba'][:, 1])
    roc_auc      = auc(fpr, tpr)
    display_name = MODEL_CONFIG[model_id]['display_name']

    ax.plot(fpr, tpr, lw=2.5, color=MODEL_COLORS[model_id],
            label=f"{display_name}  (AUC = {roc_auc:.4f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier (AUC = 0.5000)')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')

ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
ax.set_title('ROC Curves — All Models — Test Set', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Comparative Metrics Bar Chart ────────────────────────────

metrics_to_plot = {
    'accuracy':     'Accuracy',
    'auc':          'AUC-ROC',
    'f1_pneumonia': 'F1 (Pneumonia)',
    'precision':    'Precision',
    'recall':       'Recall (Sensitivity)',
}

x = np.arange(len(metrics_to_plot))
n_models  = len(MODEL_ORDER)
bar_width  = 0.18

fig, ax = plt.subplots(figsize=(16, 7))
fig.suptitle('Model Performance Comparison — Test Set', fontsize=15, fontweight='bold')

for i, model_id in enumerate(MODEL_ORDER):
    res    = EVAL_RESULTS[model_id]
    values = [res[m] for m in metrics_to_plot]
    offset = (i - n_models/2 + 0.5) * bar_width
    bars   = ax.bar(x + offset, values, bar_width,
                    label=MODEL_CONFIG[model_id]['display_name'],
                    color=MODEL_COLORS[model_id], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(list(metrics_to_plot.values()), fontsize=11)
ax.set_ylim([0.5, 1.08])
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.axhline(1.0, color='gray', lw=0.8, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ─── Detailed Classification Reports ─────────────────────────

for model_id in MODEL_ORDER:
    res = EVAL_RESULTS[model_id]
    print(f"\n{'─'*55}")
    print(f"  {MODEL_CONFIG[model_id]['display_name']} — Classification Report")
    print(f"{'─'*55}")
    print(classification_report(
        res['y_true'], res['y_pred'],
        target_names = CLASS_NAMES,
        digits       = 4
    ))


---
## 6. 🗺️ Grad-CAM XAI Visualization

Gradient-weighted Class Activation Mapping — matches `core/gradcam.py` exactly.

**SRS References**: FR-050 to FR-058
- Uses `GradientTape` to capture gradients of predicted class w.r.t. target conv layer feature maps
- Applies Global Average Pooling → ReLU → Normalize → Overlay


In [ ]:
# ─── Grad-CAM Implementation (mirrors core/gradcam.py) ─────────

def compute_gradcam_heatmap(model, target_layer_name: str,
                             img_tensor: np.ndarray,
                             pred_index: Optional[int] = None) -> np.ndarray:
    """
    Compute Grad-CAM heatmap — identical to core/gradcam.py in Streamlit app.

    SRS FR-050: GradientTape captures gradients of predicted class
    SRS FR-051: Global Average Pooling over gradient tensor
    SRS FR-052: ReLU applied to weighted feature map sum
    SRS FR-053: Normalize to [0.0, 1.0]
    """
    # Try to get target layer by name
    try:
        target_layer = model.get_layer(target_layer_name)
    except ValueError:
        # Fallback: find last conv/activation layer
        target_layer = None
        for layer in reversed(model.layers):
            if any(k in layer.name.lower() for k in ('conv', 'relu', 'activation')):
                if hasattr(layer.output, 'shape') and len(layer.output.shape) == 4:
                    target_layer = layer
                    break
        if target_layer is None:
            raise ValueError(f"Cannot locate layer: {target_layer_name}")
        print(f"  ⚠️  Fallback layer: {target_layer.name}")

    grad_model = tf.keras.Model(
        inputs  = model.inputs,
        outputs = [target_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads       = tape.gradient(class_channel, conv_outputs)
    pooled_grad = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out    = conv_outputs[0]
    heatmap     = conv_out @ pooled_grad[..., tf.newaxis]
    heatmap     = tf.squeeze(heatmap)
    heatmap     = tf.maximum(heatmap, 0)
    max_val     = tf.math.reduce_max(heatmap)
    heatmap     = heatmap / max_val if max_val > 0 else tf.zeros_like(heatmap)

    return heatmap.numpy()


def create_gradcam_overlay(orig_img_bgr: np.ndarray,
                            heatmap: np.ndarray,
                            alpha: float = 0.45,
                            colormap: int = cv2.COLORMAP_JET
                           ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Create overlay, heatmap, and composite — mirrors core/gradcam.py."""
    h, w = orig_img_bgr.shape[:2]
    hm_resized = cv2.resize(heatmap, (w, h), interpolation=cv2.INTER_CUBIC)
    hm_uint8   = np.uint8(255 * np.clip(hm_resized, 0, 1))
    hm_color   = cv2.applyColorMap(hm_uint8, colormap)
    overlay    = cv2.addWeighted(orig_img_bgr, 1-alpha, hm_color, alpha, 0)
    composite  = np.hstack([orig_img_bgr, hm_color, overlay])
    return overlay, hm_color, composite


print("✅ Grad-CAM functions defined (matches core/gradcam.py)")


In [ ]:
# ─── Generate Grad-CAM for All 4 Models ──────────────────────

def preprocess_for_inference(img_path: str, img_size: int = IMG_SIZE) -> Tuple[np.ndarray, np.ndarray]:
    """
    Preprocessing pipeline matching core/preprocessor.py exactly:
    FR-020: Grayscale | FR-021: Resize 128×128 INTER_AREA
    FR-022: 3-ch stack | FR-023: /255 | FR-024: batch expand
    """
    img_gray  = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    resized   = cv2.resize(img_gray, (img_size, img_size), interpolation=cv2.INTER_AREA)
    img_rgb   = cv2.merge([resized, resized, resized])
    normalized = img_rgb.astype(np.float32) / 255.0
    tensor    = np.expand_dims(normalized, axis=0)        # (1, 128, 128, 3)
    orig_bgr  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    return tensor, orig_bgr


# Select a PNEUMONIA sample for Grad-CAM demo
pneu_files = list((SPLIT_DIRS['test'] / 'PNEUMONIA').glob('*.jpeg'))[:1]
if not pneu_files:
    pneu_files = list((SPLIT_DIRS['test'] / 'PNEUMONIA').glob('*.jpg'))[:1]
demo_path = str(pneu_files[0]) if pneu_files else None

if demo_path:
    img_tensor, orig_bgr = preprocess_for_inference(demo_path)

    fig = plt.figure(figsize=(22, 16))
    fig.suptitle('Grad-CAM XAI — Attention Maps (PNEUMONIA Sample)
All 4 Model Architectures',
                 fontsize=15, fontweight='bold', y=1.01)

    for row_idx, model_id in enumerate(MODEL_ORDER):
        model        = LOADED_MODELS[model_id]
        target_layer = MODEL_CONFIG[model_id]['target_layer']
        display      = MODEL_CONFIG[model_id]['display_name']

        # Predict
        probs    = model.predict(img_tensor, verbose=0)[0]
        pred_idx = int(np.argmax(probs))
        conf     = float(np.max(probs)) * 100

        # Compute heatmap
        try:
            heatmap = compute_gradcam_heatmap(model, target_layer, img_tensor, pred_index=1)
            overlay, hm_color, composite = create_gradcam_overlay(orig_bgr.copy(), heatmap)
            has_cam = True
        except Exception as e:
            print(f"  ⚠️  Grad-CAM failed for {display}: {e}")
            has_cam = False

        # Row: Original | Heatmap | Overlay
        for col_idx, (panel_img, panel_title) in enumerate([
            (cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB), 'Original CXR'),
            (cv2.cvtColor(hm_color if has_cam else orig_bgr, cv2.COLOR_BGR2RGB), 'Attention Heatmap'),
            (cv2.cvtColor(overlay  if has_cam else orig_bgr, cv2.COLOR_BGR2RGB), 'Diagnostic Overlay'),
        ]):
            ax = fig.add_subplot(4, 3, row_idx * 3 + col_idx + 1)
            ax.imshow(panel_img)
            ax.set_title(f'{display}\n{panel_title}\nPred: {CLASS_LABELS[pred_idx]} ({conf:.1f}%)',
                         fontsize=9, fontweight='bold' if col_idx == 2 else 'normal')
            ax.axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'gradcam_all_models.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Grad-CAM visualization saved")
else:
    print("⚠️  No test PNEUMONIA images found for Grad-CAM demo")


---
## 7. 🎯 Ensemble Consensus Engine

**Matching `core/ensemble.py` — SRS §4.5 (FR-040 to FR-047)**

Weighted soft-voting formula (SRS §9.4):
```
P_consensus(PNEUMONIA) = 0.25·P_mobilenet + 0.35·P_densenet121 + 0.20·P_efficientnet + 0.20·P_xception
Final Verdict: PNEUMONIA if P_consensus ≥ 0.50
```


In [ ]:
# ─── Ensemble Soft-Voting Engine ─────────────────────────────

def ensemble_predict(models_dict: Dict,
                     img_tensor: np.ndarray,
                     weights: Dict) -> Dict[str, Any]:
    """
    Multi-model weighted soft-voting consensus — mirrors core/ensemble.py.
    SRS FR-041: P_consensus(C) = Sum_m [ weight_m * P_m(C) ]
    SRS FR-043: Normalize by sum of weights
    SRS FR-044: PNEUMONIA if P_consensus >= 0.50
    """
    total_weight     = 0.0
    w_prob_normal    = 0.0
    w_prob_pneumonia = 0.0
    results          = []

    for model_id, model in models_dict.items():
        w          = weights.get(model_id, 0.25)
        probs      = model.predict(img_tensor, verbose=0)[0]
        pred_idx   = int(np.argmax(probs))
        pred_label = CLASS_LABELS[pred_idx]
        conf       = float(np.max(probs)) * 100

        w_prob_normal    += w * float(probs[0])
        w_prob_pneumonia += w * float(probs[1])
        total_weight     += w

        results.append({
            'model_id':   model_id,
            'name':       MODEL_CONFIG[model_id]['display_name'],
            'weight':     w,
            'prediction': pred_label,
            'confidence': round(conf, 2),
            'prob_normal': round(float(probs[0])*100, 2),
            'prob_pneumonia': round(float(probs[1])*100, 2),
        })

    # Normalize (SRS FR-043)
    consensus_normal    = (w_prob_normal    / total_weight) * 100
    consensus_pneumonia = (w_prob_pneumonia / total_weight) * 100

    # Verdict (SRS FR-044)
    verdict    = 'PNEUMONIA' if consensus_pneumonia >= 50.0 else 'NORMAL'
    confidence = consensus_pneumonia if verdict == 'PNEUMONIA' else consensus_normal

    # Agreement (SRS FR-045)
    pneu_votes   = sum(1 for r in results if r['prediction'] == 'PNEUMONIA')
    norm_votes   = len(results) - pneu_votes
    max_votes    = max(pneu_votes, norm_votes)
    total_models = len(results)

    if max_votes == total_models:
        agreement = 'UNANIMOUS'
        agree_txt = f'Unanimous ({total_models}/{total_models})'
    elif max_votes >= total_models - 1:
        agreement = 'STRONG_MAJORITY'
        agree_txt = f'Strong Majority ({max_votes}/{total_models})'
    else:
        agreement = 'SPLIT_DECISION'
        agree_txt = f'Split Decision ({max_votes}/{total_models})'

    return {
        'verdict':     verdict,
        'confidence':  round(confidence, 2),
        'consensus_prob_pneumonia': round(consensus_pneumonia, 2),
        'consensus_prob_normal':    round(consensus_normal, 2),
        'agreement':   agreement,
        'agree_text':  agree_txt,
        'breakdown':   results,
    }


# ─── Ensemble Test Set Evaluation ────────────────────────────
print("Running ensemble evaluation on full test set...")

ENSEMBLE_WEIGHTS_MAP = {mid: MODEL_CONFIG[mid]['ensemble_weight'] for mid in MODEL_ORDER}

test_gen.reset()
all_y_true, all_ensemble_proba = [], []

# Collect individual model predictions
all_proba = {mid: [] for mid in MODEL_ORDER}
for batch_idx in tqdm(range(len(test_gen)), desc='Ensemble Inference'):
    x_batch, y_batch = test_gen[batch_idx]
    for mid in MODEL_ORDER:
        p = LOADED_MODELS[mid].predict(x_batch, verbose=0)
        all_proba[mid].append(p)
    all_y_true.extend(np.argmax(y_batch, axis=1))

# Aggregate
for mid in MODEL_ORDER:
    all_proba[mid] = np.concatenate(all_proba[mid], axis=0)

# Weighted ensemble
ensemble_proba_pneu = np.zeros(len(all_y_true))
for mid in MODEL_ORDER:
    w = ENSEMBLE_WEIGHTS_MAP[mid]
    ensemble_proba_pneu += w * all_proba[mid][:len(all_y_true), 1]
ensemble_proba_pneu /= sum(ENSEMBLE_WEIGHTS_MAP.values())

y_true_arr     = np.array(all_y_true)
y_ens_pred     = (ensemble_proba_pneu >= 0.5).astype(int)

ens_acc  = accuracy_score(y_true_arr, y_ens_pred)
ens_auc  = roc_auc_score(y_true_arr, ensemble_proba_pneu)
ens_f1   = f1_score(y_true_arr, y_ens_pred)
ens_prec = precision_score(y_true_arr, y_ens_pred)
ens_rec  = recall_score(y_true_arr, y_ens_pred)

print(f"\n{'═'*50}")
print(f"  ENSEMBLE CONSENSUS — TEST SET RESULTS")
print(f"{'═'*50}")
print(f"  Accuracy  : {ens_acc*100:.2f}%")
print(f"  AUC-ROC   : {ens_auc:.4f}")
print(f"  F1 (Pneu) : {ens_f1:.4f}")
print(f"  Precision : {ens_prec:.4f}")
print(f"  Recall    : {ens_rec:.4f}")
print(f"{'═'*50}")


In [ ]:
# ─── Ensemble Demo — Single Image ────────────────────────────

if demo_path:
    img_tensor, orig_bgr = preprocess_for_inference(demo_path)
    result = ensemble_predict(LOADED_MODELS, img_tensor, ENSEMBLE_WEIGHTS_MAP)

    print(f"\n{'═'*55}")
    print(f"  🎯 ENSEMBLE CONSENSUS RESULT")
    print(f"{'═'*55}")
    print(f"  Verdict    : {result['verdict']}")
    print(f"  Confidence : {result['confidence']:.2f}%")
    print(f"  Agreement  : {result['agree_text']}")
    print(f"  P(NORMAL)  : {result['consensus_prob_normal']:.2f}%")
    print(f"  P(PNEUMONIA): {result['consensus_prob_pneumonia']:.2f}%")
    print(f"{'─'*55}")
    print(f"  Individual Model Votes:")
    for m in result['breakdown']:
        print(f"    [{m['weight']*100:.0f}%] {m['name']:<18}: "
              f"{m['prediction']:<10} ({m['confidence']:.1f}%)")

    # Visualize ensemble probabilities
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Ensemble Consensus Demo | Verdict: {result['verdict']} "
                 f"({result['confidence']:.1f}%)", fontsize=13, fontweight='bold')

    # Bar chart of individual predictions
    model_names = [m['name'] for m in result['breakdown']]
    pneu_probs  = [m['prob_pneumonia'] for m in result['breakdown']]
    colors_bar  = ['#E74C3C' if p >= 50 else '#27AE60' for p in pneu_probs]
    bars = axes[0].bar(model_names, pneu_probs, color=colors_bar, alpha=0.8, edgecolor='white')
    axes[0].axhline(50, color='black', lw=2, linestyle='--', label='Decision threshold (50%)')
    axes[0].set_ylim([0, 105])
    axes[0].set_ylabel('P(PNEUMONIA) %', fontsize=11)
    axes[0].set_title('Individual Model Predictions', fontsize=12)
    axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
    for bar, prob in zip(bars, pneu_probs):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1.5,
                     f'{prob:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Consensus pie chart
    pies  = [result['consensus_prob_normal'], result['consensus_prob_pneumonia']]
    pie_c = ['#27AE60', '#E74C3C']
    axes[1].pie(pies, labels=['NORMAL', 'PNEUMONIA'], colors=pie_c,
                autopct='%1.1f%%', startangle=90, explode=[0.05,0.05], shadow=True)
    axes[1].set_title('Weighted Consensus Probabilities', fontsize=12)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'ensemble_demo.png', dpi=150, bbox_inches='tight')
    plt.show()


---
## 8. 🎯 YOLOv8 Pneumonia Localization

### Dataset: RSNA Pneumonia Detection Challenge
**Source**: `rsna-pneumonia-detection-challenge` on Kaggle

**Why RSNA?**
- 26,684 chest X-rays with bounding box annotations for pneumonia opacities
- PA-view radiographs (same projection as classification dataset)
- Medical expert-annotated ground truth boxes
- Perfect for teaching YOLO *where* pneumonia appears in the lung

### Pipeline:
1. Convert DICOM → PNG (1024×1024)
2. Convert RSNA format (`x, y, w, h`) → YOLO format (`cx, cy, w, h` normalized)
3. Train YOLOv8n (nano — fastest on Kaggle T4)
4. Export as `yolo_pneumonia/weights/best.pt`


In [ ]:
# ─── RSNA Dataset Inspection ──────────────────────────────────

rsna_labels_path = RSNA_DIR / 'stage_2_train_labels.csv'
rsna_class_path  = RSNA_DIR / 'stage_2_detailed_class_info.csv'
rsna_images_dir  = RSNA_DIR / 'stage_2_train_images'

# Load annotations
df_labels = pd.read_csv(rsna_labels_path)
df_class  = pd.read_csv(rsna_class_path)

print("RSNA Labels shape:", df_labels.shape)
print("Columns:", list(df_labels.columns))
print()
print(df_labels.head(10))

print(f"\nTotal annotations: {len(df_labels):,}")
print(f"Unique patients:    {df_labels['patientId'].nunique():,}")
print(f"Pneumonia cases:    {df_labels['Target'].sum():,}")
print(f"Normal cases:       {(1-df_labels['Target']).sum():,}")

# RSNA detailed class info
print("\nDetailed class info (sample):")
print(df_class['class'].value_counts())


In [ ]:
# ─── DICOM → PNG Conversion ─────────────────────────────────

# Use a balanced subset for manageable training on Kaggle
N_PNEUMONIA_TRAIN = 2500
N_NORMAL_TRAIN    = 2500
N_VAL_EACH        = 300
YOLO_IMG_SIZE     = 640   # YOLOv8 standard

# Create YOLO dataset folder structure
for split in ['train', 'val']:
    (YOLO_DATA_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATA_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print("YOLO folder structure created:")
print(f"  {YOLO_DATA_DIR}/")
print(f"  ├── images/train/  ← PNG X-rays")
print(f"  ├── images/val/")
print(f"  ├── labels/train/  ← YOLO .txt annotations")
print(f"  └── labels/val/")


def dicom_to_png(dicom_path: str, out_path: str,
                 target_size: int = YOLO_IMG_SIZE) -> Optional[Tuple[int, int]]:
    """
    Convert DICOM to PNG using VOI LUT windowing.
    Returns (original_width, original_height) or None on failure.
    """
    try:
        ds      = pydicom.dcmread(dicom_path, force=True)
        pixels  = ds.pixel_array.astype(float)

        # Apply rescale if available
        slope  = float(getattr(ds, 'RescaleSlope',  1.0))
        intcpt = float(getattr(ds, 'RescaleIntercept', 0.0))
        pixels = pixels * slope + intcpt

        # Normalize to 0-255
        p_min, p_max = pixels.min(), pixels.max()
        if p_max > p_min:
            pixels = (pixels - p_min) / (p_max - p_min) * 255.0
        pixels = pixels.astype(np.uint8)

        # Handle MONOCHROME1 (invert)
        interp = getattr(ds, 'PhotometricInterpretation', 'MONOCHROME2')
        if 'MONOCHROME1' in str(interp):
            pixels = 255 - pixels

        orig_h, orig_w = pixels.shape[:2]

        # Resize and convert to 3-channel
        img = cv2.resize(pixels, (target_size, target_size), interpolation=cv2.INTER_AREA)
        img_bgr = cv2.merge([img, img, img])
        cv2.imwrite(out_path, img_bgr)
        return orig_w, orig_h
    except Exception as e:
        return None


def rsna_to_yolo(x: float, y: float, w: float, h: float,
                 img_w: int, img_h: int) -> Tuple[float, float, float, float]:
    """
    Convert RSNA format (x, y, w, h) in pixel coords → YOLO format (cx, cy, w, h) normalized.
    x, y = top-left corner in RSNA format
    """
    cx = (x + w / 2.0) / img_w
    cy = (y + h / 2.0) / img_h
    nw = w / img_w
    nh = h / img_h
    return (
        float(np.clip(cx, 0, 1)),
        float(np.clip(cy, 0, 1)),
        float(np.clip(nw, 0, 1)),
        float(np.clip(nh, 0, 1)),
    )


print("✅ DICOM conversion utilities defined")


In [ ]:
# ─── Build YOLO Dataset from RSNA ────────────────────────────

# Identify pneumonia and normal patients
pneumonia_ids = df_labels[df_labels['Target'] == 1]['patientId'].unique().tolist()
normal_ids    = df_labels[df_labels['Target'] == 0]['patientId'].unique().tolist()

random.shuffle(pneumonia_ids)
random.shuffle(normal_ids)

# Split into train / val
pneu_train = pneumonia_ids[:N_PNEUMONIA_TRAIN]
pneu_val   = pneumonia_ids[N_PNEUMONIA_TRAIN:N_PNEUMONIA_TRAIN + N_VAL_EACH]
norm_train = normal_ids[:N_NORMAL_TRAIN]
norm_val   = normal_ids[N_NORMAL_TRAIN:N_NORMAL_TRAIN + N_VAL_EACH]

split_map = {
    'train': pneu_train + norm_train,
    'val':   pneu_val   + norm_val,
}

RSNA_ORIG_SIZE = 1024   # All RSNA images are 1024×1024

skipped = 0
processed = {'train': 0, 'val': 0}

for split_name, patient_ids in split_map.items():
    img_out_dir = YOLO_DATA_DIR / 'images' / split_name
    lbl_out_dir = YOLO_DATA_DIR / 'labels' / split_name

    for pid in tqdm(patient_ids, desc=f'Processing {split_name}'):
        dicom_path = rsna_images_dir / f'{pid}.dcm'
        if not dicom_path.exists():
            skipped += 1
            continue

        out_png  = str(img_out_dir / f'{pid}.png')
        orig_dim = dicom_to_png(str(dicom_path), out_png, YOLO_IMG_SIZE)

        if orig_dim is None:
            skipped += 1
            continue

        orig_w, orig_h = orig_dim  # Usually 1024×1024

        # Build YOLO label
        patient_boxes = df_labels[
            (df_labels['patientId'] == pid) & (df_labels['Target'] == 1)
        ]

        label_lines = []
        for _, row in patient_boxes.iterrows():
            if pd.notna(row['x']) and row['Target'] == 1:
                cx, cy, nw, nh = rsna_to_yolo(
                    float(row['x']), float(row['y']),
                    float(row['width']), float(row['height']),
                    orig_w, orig_h
                )
                label_lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        # Write label file (empty file = normal/background image)
        lbl_path = lbl_out_dir / f'{pid}.txt'
        with open(lbl_path, 'w') as f:
            f.write('\n'.join(label_lines))

        processed[split_name] += 1

print(f"\nDataset built:")
print(f"  Train: {processed['train']:,} images")
print(f"  Val  : {processed['val']:,} images")
print(f"  Skip : {skipped:,} (missing DICOM files)")


In [ ]:
# ─── Create YOLO dataset.yaml ─────────────────────────────────

dataset_yaml_content = f"""# Pneumonia Detection Dataset
# Generated from RSNA Pneumonia Detection Challenge
# Classes: 0 = pneumonia_opacity

path: {YOLO_DATA_DIR}
train: images/train
val:   images/val

nc: 1
names:
  0: pneumonia_opacity
"""

yaml_path = YOLO_DATA_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    f.write(dataset_yaml_content)

print("✅ dataset.yaml created:")
print(dataset_yaml_content)

# Verify dataset
n_train_imgs = len(list((YOLO_DATA_DIR / 'images' / 'train').glob('*.png')))
n_val_imgs   = len(list((YOLO_DATA_DIR / 'images' / 'val').glob('*.png')))
n_train_lbls = len(list((YOLO_DATA_DIR / 'labels' / 'train').glob('*.txt')))
n_val_lbls   = len(list((YOLO_DATA_DIR / 'labels' / 'val').glob('*.txt')))

# Count labels with annotations
def count_annotated(lbl_dir):
    annotated = sum(1 for f in Path(lbl_dir).glob('*.txt')
                    if f.stat().st_size > 0)
    return annotated

print(f"\nDataset Verification:")
print(f"  Train: {n_train_imgs:,} images | {n_train_lbls:,} labels "
      f"| {count_annotated(YOLO_DATA_DIR/'labels'/'train'):,} annotated")
print(f"  Val:   {n_val_imgs:,} images | {n_val_lbls:,} labels "
      f"| {count_annotated(YOLO_DATA_DIR/'labels'/'val'):,} annotated")


In [ ]:
# ─── YOLOv8 Training ──────────────────────────────────────────
# Using YOLOv8n (nano) — fastest on T4, good accuracy for medical imaging

print("=" * 60)
print("  🎯 TRAINING YOLOv8n — PNEUMONIA LOCALIZATION")
print("=" * 60)
print(f"  Dataset  : RSNA Pneumonia Detection Challenge")
print(f"  Train    : {n_train_imgs:,} images")
print(f"  Val      : {n_val_imgs:,} images")
print(f"  Classes  : 1 (pneumonia_opacity)")
print(f"  Img size : {YOLO_IMG_SIZE}")
print(f"  Backbone : YOLOv8n (nano, pre-trained COCO)")
print("=" * 60)

# Initialize YOLOv8n with COCO pretrained weights
yolo_model = YOLO('yolov8n.pt')

# Train
yolo_results = yolo_model.train(
    data         = str(yaml_path),
    epochs       = 50,              # Use 80+ for production
    imgsz        = YOLO_IMG_SIZE,
    batch        = 16,
    lr0          = 0.01,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 3,
    patience     = 15,             # Early stopping
    device       = 0,              # GPU
    workers      = 2,
    project      = str(OUTPUT_DIR / 'yolo_runs'),
    name         = 'yolo_pneumonia',
    exist_ok     = True,
    save         = True,
    save_period  = 10,             # Save checkpoint every 10 epochs
    plots        = True,
    verbose      = True,
    # Medical imaging augmentation
    augment      = True,
    degrees      = 5.0,            # Rotation ±5°
    translate    = 0.1,
    scale        = 0.1,
    fliplr       = 0.5,            # Horizontal flip (valid for CXR)
    flipud       = 0.0,            # No vertical flip (anatomically incorrect)
    mosaic       = 0.5,
    close_mosaic = 10,
    hsv_h        = 0.0,            # No hue shift (grayscale)
    hsv_s        = 0.0,
    hsv_v        = 0.2,            # Brightness variation
)

print("\n✅ YOLOv8 training complete!")


In [ ]:
# ─── YOLO Model Evaluation ─────────────────────────────────

# Best model path
yolo_best_path = OUTPUT_DIR / 'yolo_runs' / 'yolo_pneumonia' / 'weights' / 'best.pt'

if yolo_best_path.exists():
    print(f"✅ Best model: {yolo_best_path}")
    best_yolo = YOLO(str(yolo_best_path))

    # Validate
    val_metrics = best_yolo.val(
        data    = str(yaml_path),
        imgsz   = YOLO_IMG_SIZE,
        device  = 0,
        verbose = True,
    )

    print(f"\n{'='*55}")
    print(f"  YOLOv8 VALIDATION METRICS")
    print(f"{'='*55}")
    try:
        print(f"  mAP@0.5     : {val_metrics.box.map50:.4f}")
        print(f"  mAP@0.5:0.95: {val_metrics.box.map:.4f}")
        print(f"  Precision   : {val_metrics.box.p.mean():.4f}")
        print(f"  Recall      : {val_metrics.box.r.mean():.4f}")
    except Exception as e:
        print(f"  Metrics: {e}")
    print(f"{'='*55}")
else:
    print(f"⚠️  Best model not found at {yolo_best_path}")
    print("   Checking alternative paths...")
    alt_paths = list((OUTPUT_DIR / 'yolo_runs').rglob('best.pt'))
    if alt_paths:
        yolo_best_path = alt_paths[0]
        best_yolo = YOLO(str(yolo_best_path))
        print(f"   Found: {yolo_best_path}")


In [ ]:
# ─── YOLO Inference Visualization ────────────────────────────

# Run inference on sample validation images
sample_val_imgs = list((YOLO_DATA_DIR / 'images' / 'val').glob('*.png'))[:6]

if sample_val_imgs and 'best_yolo' in dir():
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('YOLOv8 Pneumonia Localization — Validation Samples',
                 fontsize=15, fontweight='bold')

    for idx, img_path in enumerate(sample_val_imgs[:6]):
        row, col = divmod(idx, 3)
        ax = axes[row][col]

        # Read image
        orig = cv2.imread(str(img_path))
        orig_rgb = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)

        # Load GT label
        lbl_path = YOLO_DATA_DIR / 'labels' / 'val' / (img_path.stem + '.txt')
        gt_boxes = []
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f.read().strip().split('\n'):
                    if line:
                        parts = line.split()
                        if len(parts) == 5:
                            _, cx, cy, nw, nh = [float(x) for x in parts]
                            H, W = orig.shape[:2]
                            x1 = int((cx - nw/2) * W)
                            y1 = int((cy - nh/2) * H)
                            x2 = int((cx + nw/2) * W)
                            y2 = int((cy + nh/2) * H)
                            gt_boxes.append((x1, y1, x2, y2))

        # YOLO prediction
        results = best_yolo.predict(str(img_path), imgsz=YOLO_IMG_SIZE, conf=0.25, verbose=False)
        pred_img = orig_rgb.copy()

        # Draw GT boxes (green)
        for (x1, y1, x2, y2) in gt_boxes:
            cv2.rectangle(pred_img, (x1,y1), (x2,y2), (0,200,0), 3)
            cv2.putText(pred_img, 'GT', (x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,200,0), 2)

        # Draw predicted boxes (red)
        result = results[0]
        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                x1,y1,x2,y2 = [int(v) for v in box.xyxy[0]]
                conf_val = float(box.conf[0])
                cv2.rectangle(pred_img, (x1,y1), (x2,y2), (255,50,50), 2)
                cv2.putText(pred_img, f'Pred {conf_val:.2f}',
                            (x1, y2+20), cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (255,50,50), 2)

        ax.imshow(pred_img)
        has_pneumonia = len(gt_boxes) > 0
        ax.set_title(f"{'PNEUMONIA' if has_pneumonia else 'NORMAL'}\n"
                     f"GT boxes: {len(gt_boxes)} | Pred boxes: "
                     f"{len(result.boxes) if result.boxes else 0}",
                     fontsize=9, color='#E74C3C' if has_pneumonia else '#27AE60')
        ax.axis('off')

    # Legend
    from matplotlib.patches import Patch
    legend_elems = [Patch(facecolor='#00C800', label='Ground Truth'),
                    Patch(facecolor='#FF3232', label='YOLOv8 Prediction')]
    fig.legend(handles=legend_elems, loc='lower center', ncol=2, fontsize=12,
               bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'yolo_inference_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 YOLO inference visualization saved")


---
## 9. 💾 Model Export & Streamlit Integration

### Download Instructions
After running this notebook, download these files from `/kaggle/working/`:

| File | Model | Upload To |
|------|-------|-----------|
| `mobilenet_model.h5` | MobileNetV2 | Hugging Face Space (Git LFS) |
| `efficientnet_model.h5` | EfficientNetB0 | Hugging Face Space (Git LFS) |
| `densenet121_model.h5` | DenseNet121 | Hugging Face Space (Git LFS) |
| `xception_model.h5` | Xception | Hugging Face Space (Git LFS) |
| `yolo_pneumonia/weights/best.pt` | YOLOv8n | Streamlit app `models/` folder |

### Streamlit App Integration for YOLO
Add the YOLO inference to your Streamlit app by calling `best.pt` with `ultralytics`.


In [ ]:
# ─── Verify All Saved Model Files ────────────────────────────

print("=" * 65)
print("  📦 EXPORTED MODEL FILES — VERIFICATION")
print("=" * 65)

all_good = True

# 1. Classification models (.h5)
print("\n🔬 CNN Classification Models (.h5):")
print(f"{'File':<30} {'Size':>10} {'Status':>10}")
print("─" * 55)

for model_id in MODEL_ORDER:
    filename = MODEL_CONFIG[model_id]['filename']
    path = OUTPUT_DIR / filename
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f"{filename:<30} {size_mb:>9.1f}M  {'✅ OK':>8}")
    else:
        print(f"{filename:<30} {'':>10} {'❌ MISSING':>10}")
        all_good = False

# 2. YOLO model
print("\n🎯 YOLOv8 Localization Model:")
print(f"{'File':<30} {'Size':>10} {'Status':>10}")
print("─" * 55)

yolo_paths = [
    OUTPUT_DIR / 'yolo_runs' / 'yolo_pneumonia' / 'weights' / 'best.pt',
    OUTPUT_DIR / 'yolo_pneumonia' / 'weights' / 'best.pt',
]
yolo_paths += list((OUTPUT_DIR / 'yolo_runs').rglob('best.pt')) if (OUTPUT_DIR/'yolo_runs').exists() else []

yolo_found = False
for p in yolo_paths:
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f"{'best.pt (YOLO)':<30} {size_mb:>9.1f}M  {'✅ OK':>8}")
        print(f"  Path: {p}")
        yolo_found = True
        break

if not yolo_found:
    print(f"{'best.pt (YOLO)':<30} {'':>10} {'❌ MISSING':>10}")
    all_good = False

# 3. Evaluation assets
print("\n📊 Evaluation Assets:")
eval_files = [
    'eda_class_distribution.png',
    'eda_sample_images.png',
    'training_histories.png',
    'confusion_matrices.png',
    'roc_curves.png',
    'metrics_comparison.png',
    'model_comparison.csv',
    'gradcam_all_models.png',
    'yolo_inference_samples.png',
]
for f in eval_files:
    p = OUTPUT_DIR / f
    status = '✅' if p.exists() else '⚠️ '
    print(f"  {status} {f}")

print("\n" + "=" * 65)
if all_good:
    print("  ✅  ALL MODELS EXPORTED SUCCESSFULLY — Ready for Streamlit!")
else:
    print("  ⚠️   Some files missing — check training logs above")
print("=" * 65)


In [ ]:
# ─── YOLO Integration Code for Streamlit App ─────────────────
# 
# Add this to your streamlit_app.py or core/ module
#
# NOTE: Your existing CNN models are already fully integrated via config.py
#       This code adds the NEW YOLO localization feature

yolo_integration_code = '''
# === YOLO Integration (add to streamlit_app.py) ===

from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image

# Load YOLO model (add to your ModelManager or init)
YOLO_MODEL_PATH = "models/yolo_pneumonia_best.pt"

@st.cache_resource
def load_yolo_model():
    """Load YOLOv8 localization model (cached)."""
    try:
        return YOLO(YOLO_MODEL_PATH)
    except Exception as e:
        st.warning(f"YOLO model not available: {e}")
        return None


def run_yolo_localization(image_array: np.ndarray,
                           yolo_model,
                           conf_threshold: float = 0.25) -> dict:
    """
    Run YOLOv8 pneumonia localization on a chest X-ray.
    
    Args:
        image_array: NumPy array (H, W, 3) BGR or RGB
        yolo_model: Loaded YOLO model
        conf_threshold: Minimum confidence for detection
        
    Returns:
        dict with 'has_detection', 'boxes', 'annotated_image'
    """
    if yolo_model is None:
        return {'has_detection': False, 'boxes': [], 'annotated_image': image_array}
    
    # Resize to YOLO training size
    img_resized = cv2.resize(image_array, (640, 640))
    
    results = yolo_model.predict(img_resized, conf=conf_threshold, verbose=False)
    result  = results[0]
    
    annotated = img_resized.copy()
    boxes_info = []
    
    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes:
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]
            conf = float(box.conf[0])
            
            # Draw box
            cv2.rectangle(annotated, (x1,y1), (x2,y2), (255, 50, 50), 3)
            label = f"Pneumonia {conf:.1%}"
            cv2.rectangle(annotated, (x1, y1-30), (x1+len(label)*9, y1), (255,50,50), -1)
            cv2.putText(annotated, label, (x1+3, y1-8),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
            
            boxes_info.append({
                'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                'confidence': conf
            })
    
    return {
        'has_detection': len(boxes_info) > 0,
        'boxes': boxes_info,
        'annotated_image': annotated,
    }


# In your main Streamlit UI:
# yolo_model = load_yolo_model()
# yolo_result = run_yolo_localization(uploaded_image_array, yolo_model)
# if yolo_result['has_detection']:
#     st.image(yolo_result['annotated_image'], caption="YOLO Localization")
#     for box in yolo_result['boxes']:
#         st.info(f"Pneumonia region detected: confidence {box['confidence']:.1%}")
'''

print("YOLO Streamlit Integration Code:")
print("─" * 55)
print(yolo_integration_code)

# Save to file
with open(OUTPUT_DIR / 'yolo_streamlit_integration.py', 'w') as f:
    f.write(yolo_integration_code)
print("\n📄 yolo_streamlit_integration.py saved to /kaggle/working/")


In [ ]:
# ─── Final Summary Report ─────────────────────────────────────

BORDER = '=' * 62
print('\n' + BORDER)
print('  PNEUMONIA DIAGNOSTIC HUB -- TRAINING COMPLETE')
print(BORDER)

print(f"\n📊 CLASSIFICATION MODELS (4 CNNs):")
print(f"{'─'*60}")
print(f"{'Model':<22} {'Test Acc':>10} {'AUC':>8} {'F1':>8} {'Weight':>8}")
print(f"{'─'*60}")
for mid in MODEL_ORDER:
    res = EVAL_RESULTS[mid]
    print(f"{MODEL_CONFIG[mid]['display_name']:<22} "
          f"{res['accuracy']*100:>9.2f}% "
          f"{res['auc']:>8.4f} "
          f"{res['f1_pneumonia']:>8.4f} "
          f"{MODEL_CONFIG[mid]['ensemble_weight']*100:>7.0f}%")

print(f"{'─'*60}")
print(f"{'Ensemble Consensus':<22} {ens_acc*100:>9.2f}% {ens_auc:>8.4f} {ens_f1:>8.4f} {'100%':>8}")

print(f"\n🎯 YOLO LOCALIZATION MODEL:")
print(f"   Architecture : YOLOv8n")
print(f"   Dataset      : RSNA Pneumonia Detection Challenge")
print(f"   Train images : {n_train_imgs:,}")
print(f"   Val images   : {n_val_imgs:,}")

print(f"\n💾 OUTPUT FILES (/kaggle/working/):")
output_files = list(OUTPUT_DIR.glob('*.h5')) + list(OUTPUT_DIR.rglob('best.pt'))
for f in sorted(output_files):
    size_mb = f.stat().st_size / 1e6
    print(f"   📁 {f.name:<35} {size_mb:.1f} MB")

print('
' + '=' * 62)
print('  NEXT STEPS -- STREAMLIT DEPLOYMENT')
print('=' * 62)
print('  1. Download .h5 files from /kaggle/working/')
print('  2. Upload to Hugging Face Spaces (Git LFS for large files)')
print('  3. Download best.pt and place in models/ in your app')
print('  4. Add yolo_streamlit_integration.py code to streamlit_app')
print("  5. Your app's config.py already has the right model names!")
print('=' * 62)
print('Training complete!')
